<a href="https://colab.research.google.com/github/tharun0223darla/flyrank-search-intelligence/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tharun0223darla/flyrank-search-intelligence/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Skill: Framing ML Problems (`framing-ml-problems/SKILL.md`)

This skill helps you define an ML problem clearly.

### 1. My lane as an ML task (type)
*   **Classification:** Predict a category (e.g., spam/not spam, disease/no disease).
*   **Regression:** Predict a continuous value (e.g., house price, temperature).
*   **Clustering:** Group similar items together (e.g., customer segmentation).
*   **Ranking:** Order items by relevance (e.g., search results, product recommendations).
*   **Scoring:** Assign a score to an item (e.g., credit score, risk score).

Choose one that best fits what you want your model to do.

### 2. Target or proxy

*   **Target:** The direct outcome you want to predict. This should ideally be observed data.
*   **Proxy:** An indirect measure that you predict when the true target is unavailable or too costly to obtain directly. Ensure your proxy is strongly correlated with the true target.

Clearly define how this label is generated (observed outcome, business rule, human annotation, etc.).

### 3. Success metric

*   **Choose one primary metric** that directly reflects the business goal.
*   **Defend your choice:** Explain why this metric is appropriate and what a 'good' value for it would be (e.g., 90% accuracy, AUC > 0.8, average rank improvement).
*   Consider metrics like precision, recall, F1-score, AUC, RMSE, MAE, R-squared, NDCG, etc.

### 4. The unit of analysis, as a real dataframe

*   **Identify the 'one thing' your model makes a prediction about.** This is your unit of analysis.
*   **Show a sample dataframe:** Load a small slice of your data (`df.head()` or `df.sample()`) where each row represents one unit of analysis.
*   **Describe what each row represents** (e.g., 'one row = one user-query pair', 'one row = one product', 'one row = one customer session').

### 5. Why ML beats a fixed rule here

*   **Explain the complexity:** Describe why a simple `if-then-else` rule or a static threshold is insufficient or too brittle.
*   **Highlight patterns:** Point out that the relationships are too subtle, non-linear, numerous, or dynamic for a hand-coded rule.
*   **Focus on scale or adaptability:** ML can handle large numbers of features, discover hidden patterns, and adapt to changing data more effectively than fixed rules.

## Skill: FlyRank Data (`flyrank/flyrank-data/SKILL.md`)

This skill provides context and guidance for working with the FlyRank dataset.

### Data Source
The FlyRank dataset is typically sourced from a BigQuery table. You'll often use `pandas_gbq` to load the data.

### Key Tables/Concepts
*   **Impressions:** Records of when a search result was shown to a user.
    *   `session_id`: Unique identifier for a user session.
    *   `query_id`: Unique identifier for a search query within a session.
    *   `item_id`: Unique identifier for the item displayed.
    *   `display_rank`: The position of the item in the search results.
    *   `timestamp`: When the impression occurred.
*   **Clicks:** Records of when a user clicked on a displayed item.
    *   `session_id`, `query_id`, `item_id`, `timestamp`: Corresponding to the impression.
*   **Features:** Information about queries, items, and users.
    *   `query_features`: e.g., query length, number of keywords, categorization.
    *   `item_features`: e.g., item description length, price, category, historical popularity.
    *   `user_features`: e.g., historical clicks, demographics (if available).

### Common Data Preparation Steps
1.  **Load Data:** Use `pandas_gbq` to query relevant tables from BigQuery.
2.  **Join Data:** Combine impressions, clicks, and feature data using `session_id`, `query_id`, and `item_id`.
3.  **Define Labels:** For ranking tasks, a common label is whether an item was clicked after an impression. You might also consider conversion events if available.
    *   `is_clicked`: A binary (0/1) label indicating a click.
4.  **Feature Engineering:** Create new features from existing ones.
    *   **Interaction features:** e.g., `query_item_match_score`.
    *   **Temporal features:** e.g., `time_since_last_click`.
    *   **Aggregations:** e.g., `item_historical_click_through_rate`.
5.  **Handle Missing Values:** Decide on imputation strategies (mean, median, mode) or removal.
6.  **Encoding Categorical Features:** Use one-hot encoding or other methods for categorical variables.
7.  **Scaling Numerical Features:** Standardize or normalize numerical features if using distance-based models.

### Unit of Analysis for Ranking
For ranking tasks, the typical unit of analysis is a **query-item pair** at the time of impression. Each row in your training data will represent a single item shown for a specific query, with features describing the query, the item, and their interaction, along with the click label.

## 1. My lane as an ML task (type)

This problem is a **Ranking** task. We want to predict the relevance of items (e.g., search results) given a query, and order them such that more relevant items appear higher. This aligns with the 'FlyRank' context which aims to improve search result ordering.

## 2. Target or proxy

The target we want to predict is whether an `item` is **clicked** by a user after being displayed for a given `query`. This is a direct **observed outcome** from user interactions (clicks) after an impression. We will create a binary label (`is_clicked`: 1 for clicked, 0 for not clicked) by joining impression data with click data. If an impression has a corresponding click, `is_clicked` is 1; otherwise, it's 0.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

The primary success metric will be **Normalized Discounted Cumulative Gain (NDCG)**. NDCG is well-suited for ranking problems because it takes into account the position of relevant items. It penalizes highly relevant items appearing lower in the ranking more than less relevant items. A 'good' value for NDCG would depend on the baseline performance, but we would aim for a value closer to 1.0, indicating a highly effective ranking. For example, an NDCG score above 0.8 would likely be considered strong improvement over a random or simple baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is a **query-item pair**. Each row in our dataframe will represent a specific item shown for a specific query within a user session, along with features about the query, the item, the user, and their interaction, ultimately with a binary `is_clicked` label. This structure allows the model to learn what makes an item relevant to a query in a given context.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Placeholder for loading data and showing the unit of analysis
# You would typically query BigQuery for impressions and clicks,
# then join them to create your unit of analysis.

# Example (replace with actual data loading and processing):
# import pandas as pd
# import pandas_gbq

# BQ_PROJECT = 'your-gcp-project-id' # Make sure to define this variable

# # Example: Load a small sample of impression data
# query_impressions = """
# SELECT session_id, query_id, item_id, display_rank, timestamp
# FROM `turing-g-cloud-da.nl2sql.flyrank_impressions`
# LIMIT 100
# """
# impressions_df = pandas_gbq.read_gbq(query_impressions, project_id=BQ_PROJECT)

# # Example: Load a small sample of click data
# query_clicks = """
# SELECT session_id, query_id, item_id, timestamp
# FROM `turing-g-cloud-da.nl2sql.flyrank_clicks`
# LIMIT 100
# """
# clicks_df = pandas_gbq.read_gbq(query_clicks, project_id=BQ_PROJECT)

# # Example: Join to create unit of analysis and add is_clicked label
# # (Simplified for demonstration, actual join logic might be more complex)
# unit_of_analysis_df = impressions_df.merge(
#     clicks_df.assign(is_clicked=1), # Assign 1 to clicked items
#     on=['session_id', 'query_id', 'item_id'],
#     how='left', # Keep all impressions
#     suffixes=('_impression', '_click')
# ).fillna({'is_clicked': 0})

# # Display the head of the resulting dataframe
# display(unit_of_analysis_df.head())

# # Describe what one row represents:
# # print("Each row represents a unique (session_id, query_id, item_id) pair, indicating an impression.")
# # print("The 'is_clicked' column shows whether that item was clicked for that impression.")

## 5. Why ML beats a fixed rule here

The pattern for search result relevance is far too complex and dynamic for a fixed rule. Consider the multitude of factors influencing a click:

1.  **User Intent:** The same query can have different intents (e.g., 'apple' as a fruit vs. a company). A fixed rule cannot easily adapt.
2.  **Item Attributes:** Item relevance depends on many features (price, description, reviews, category) and their interactions, which are hard to capture with simple `if-then` statements.
3.  **Contextual Factors:** Time of day, user's past behavior, current trends, and geographic location all influence relevance. Rules struggle with such high-dimensional, transient context.
4.  **Evolving Data:** New items are added constantly, and user preferences change. Hand-coded rules would require continuous, manual updates.
5.  **Subtle Relationships:** The optimal ranking often involves non-linear and intricate relationships between features that are impossible for humans to define exhaustively in a rule set. ML models can discover these subtle patterns from data, adapting to new information and improving performance over time in a way that fixed rules cannot.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.